# Person linkage — stage 1: from a table of names to a ranked worklist

**What this notebook is.** The first working session of the person record-linkage
project: over 350 years the same human being appears in many acts, and the database
has no way to say *these rows are one person*. The plan, the evidence behind it and
the design decisions live in
[`docs/person_linkage/research.md`](../docs/person_linkage/research.md); this notebook
is the lab record of carrying it out.

**Scope of this notebook.** The whole of stage 1: build the **person spine** (one row
per person carrying every piece of evidence), set aside the rows that are not people,
let deterministic rules settle what they can, and train a probabilistic model to rank
the rest. The output is a worklist for historians — **nothing here writes to the
database**, and no machine decision is final.

**How to read it.** Each section asks a question, runs a short piece of code, and
then says in plain language what the result means. You should be able to follow the
argument without reading any code. Sections end with `assert` checkpoints — over the
data in §6, and over the trained **model** in §9 — so that if either drifts, this
notebook fails loudly instead of quietly changing its conclusions. The model
checkpoint was added on 2026-08-29 after an audit found that the model shipped the
previous day was not a valid one and nothing here had noticed; §9 and the closing
section explain what happened and why every figure below changed.

**How to run it.**

```
uv sync --extra linkage
uv run --extra linkage jupyter lab
```

## Setup

We open the corpus read-only and load the spine builder. The heavy lifting lives in `workflows/person_features.py` — tested code, imported here — so that this notebook and the production batch can never drift apart.

In [1]:
import sys, pandas as pd, altair as alt
sys.path.insert(0, "..")

from workflows.person_features import (
    load_person_spine, open_ro, normalize_name, HUB_CONTRACT_THRESHOLD,
)

# Charts are stored as HTML in the committed notebook (altair's default), so
# they survive a re-read without re-running. Drawing them fetches the Vega
# library from a CDN, so an offline reader sees the spec but no picture.
alt.renderers.enable("default")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

conn = open_ro()          # mode=ro: this notebook cannot write to the corpus
print("corpus opened read-only ·  hub threshold =", HUB_CONTRACT_THRESHOLD, "contracts")

corpus opened read-only ·  hub threshold = 20 contracts


## 1. Who is actually in the `person` table?

Before building features, we need to know how much evidence exists to build them
*from*. The single most important question for a linkage model is not "how many
people" but "how often is each field actually filled in" — a field that is missing
half the time behaves very differently from one that is always there.

In [2]:
people = pd.read_sql("SELECT * FROM person WHERE is_deleted = 0", conn)

fill = pd.DataFrame({
    "filled": [(people[c].fillna("").astype(str).str.strip() != "").sum()
               for c in ["first_name", "father_mother", "grandfather", "last_name", "nickname"]],
}, index=["first name", "patronymic (father)", "grandfather", "surname", "nickname"])
fill["% of persons"] = (100 * fill["filled"] / len(people)).round(1)

print(f"live persons: {len(people):,}   ·   recorded as women: {int(people.is_woman.sum())}")
fill

live persons: 11,391   ·   recorded as women: 242


,filled,% of persons
first name,11216,98.5
patronymic (father),6437,56.5
grandfather,1445,12.7
surname,10855,95.3
nickname,41,0.4


**What we see.** Names are nearly always present, but the *distinguishing* parts
are not. The patronymic — "di Filippo", the single most useful field for telling two
same-named men apart — is missing for **43%** of people, and the grandfather for
**87%**. This one table dictates the shape of the whole model: it must handle
"missing" as its own state, distinct from "different". Two men who both lack a
patronymic have not thereby *agreed* about anything.

It also explains why the network and time features matter so much here: for nearly
half the corpus, the name fields alone simply run out of information.

## 2. Building the spine

The spine is one row per person with every model input attached: the verbatim name
fields, their normalized twins (used only as comparison keys — the originals are
never altered), how often and when they appear, who they invested alongside, and in
what role.

In [3]:
spine = load_person_spine(conn)
print(f"{len(spine):,} persons  ×  {len(spine.columns)} columns")
list(spine.columns)

11,391 persons  ×  30 columns


['person_id',
 'first_name',
 'father_mother',
 'grandfather',
 'last_name',
 'nickname',
 'is_woman',
 'first_norm',
 'patronymic_norm',
 'grandfather_norm',
 'last_norm',
 'entity_kind',
 'full_name_norm',
 'n_appearances',
 'n_posthumous',
 'contracts',
 'first_year',
 'last_year',
 'n_dated',
 'span_years',
 'first_posthumous_year',
 'partners',
 'n_partners',
 'firm_tokens',
 'firms',
 'n_gp',
 'n_lp',
 'dominant_role',
 'husband_first_norm',
 'husband_last_norm']

**What we see.** 11,391 people, each with 28 columns of evidence. Three groups
matter most, and each encodes a historical judgement rather than a mechanical
transformation:

| Group | Columns | The judgement inside it |
|---|---|---|
| Identity | `first_norm`, `patronymic_norm`, `grandfather_norm`, `last_norm` | folds only what is scribal (accents, the three stored apostrophes) — never what is meaningful (`Rossi` ≠ `Rosso`, `Francesco` ≠ `Francesca`) |
| Time | `first_year`, `last_year`, `span_years`, `first_posthumous_year` | a career is what someone did **while alive** |
| Network | `partners`, `firm_tokens` | sharing a rare partner is evidence; sharing a famous one is not |

The next three sections examine exactly those three judgements.

## 3. Meet the cast

Rather than abstract statistics, we follow the same handful of real people through
the whole notebook. Each was identified during the research phase and each stands
for one of the four faces of the problem.

In [4]:
CAST = {
    1641:  "Strozzi — probably the same man as 10746 (under-linked)",
    10746: "Strozzi — his career picks up three years after 1641's ends",
    4806:  "Strozzi — a different man: grandfather Alfonso, a century later",
    1164:  "Guicciardini — one row holding two lives (over-linked)",
    12205: "Corradini — a batch-entry ghost, never appears anywhere",
    12208: "Corradini — the same name, and the one who actually appears",
    3460:  "Tornaquinci — invests in 1599, his estate acts in 1642",
}
cols = ["person_id", "first_name", "father_mother", "grandfather", "last_name",
        "n_appearances", "first_year", "last_year", "span_years",
        "first_posthumous_year", "n_partners", "dominant_role"]
spine[spine.person_id.isin(CAST)][cols].set_index("person_id")

,first_name,father_mother,grandfather,last_name,n_appearances,first_year,last_year,span_years,first_posthumous_year,n_partners,dominant_role
person_id,,,,,,,,,,,
1164,Agnolo,Girolamo,Agnolo,Guicciardini,13,1559.0,1628.0,69.0,NaN,21,lp
1641,Alfonso,Filippo,Matteo,Strozzi,2,1491.0,1517.0,26.0,NaN,5,mixed
3460,Mario,NaN,NaN,Tornaquinci,2,1599.0,1599.0,0.0,1642.0,15,mixed
4806,Alfonso,Filippo,Alfonso,Strozzi,1,1620.0,1620.0,0.0,NaN,4,lp
10746,Alfonso,Filippo,NaN,Strozzi,2,1520.0,1526.0,6.0,NaN,4,lp
12205,Lisabetta,Giovanni,Bartolomeo,Corradini,0,NaN,NaN,NaN,NaN,0,unknown
12208,Lisabetta,Giovanni,Bartolomeo,Corradini,1,1590.0,1590.0,0.0,NaN,2,lp


**What we see.** The table already tells four different stories:

- **The two Alfonso Strozzi rows (1641, 10746)** have identical names and careers that
  abut — 1491–1517, then 1520–1526. That is what an under-linked person looks like.
- **The third Alfonso (4806)** looks identical too, but appears in 1620 and records a
  *different* grandfather. Same name, different man, a century apart.
- **Agnolo Guicciardini (1164)** spans **69 years** in a single row. No merchant career
  runs that long; this is a grandfather and a grandson fused together, which the
  Florentine habit of naming the eldest son after his grandfather makes almost
  invisible.
- **The two Lisabetta Corradini rows** differ in one respect only: one appears in a
  contract, the other appears nowhere at all.

## 4. What counts as a career?

A person's activity window — first to last appearance — is the one feature available
for essentially everybody, so its definition carries a lot of weight. Two kinds of
row would quietly corrupt it, and the spine excludes both.

In [5]:
mario = spine[spine.person_id == 3460].iloc[0]
print("Mario Tornaquinci (person 3460)")
print(f"  appearances               : {mario.n_appearances}  "
      f"(of which posthumous: {mario.n_posthumous})")
print(f"  career window as recorded : {mario.first_year:.0f}-{mario.last_year:.0f}  "
      f"→ span {mario.span_years:.0f} years")
print(f"  estate first acts in      : {mario.first_posthumous_year:.0f}")
print(f"  naive window (all rows)   : 1599-1642 → span 43 years")

Mario Tornaquinci (person 3460)
  appearances               : 2  (of which posthumous: 1)
  career window as recorded : 1599-1599  → span 0 years
  estate first acts in      : 1642
  naive window (all rows)   : 1599-1642 → span 43 years


**What we see.** Mario Tornaquinci invested once, in 1599. In 1642 his *heirs*
invest in his name. Counting that as part of his career would stretch his working
life by 43 years and make him look like an over-merge. So posthumous rows are
excluded from the window — but not discarded: the year his estate first acts is kept
separately, because it is evidence of something else entirely, namely that he had
died by then.

The same applies to the 20 contracts dated `0000-00-00`: treated naively they produce
careers spanning millennia. A person whose only appearance is undated therefore gets
**no** window rather than an absurd one.

In [6]:
dated = spine[spine.n_dated > 0]
heirs_only = spine[(spine.n_dated == 0) & (spine.n_posthumous > 0)]
print(f"persons with a living, dated career : {len(dated):,}")
print(f"persons known only posthumously     : {len(heirs_only):,}")
print(f"                              total : {len(dated) + len(heirs_only):,}"
      f"   ← the memo's 10,527 counts both")

persons with a living, dated career : 10,425
persons known only posthumously     : 102
                              total : 10,527   ← the memo's 10,527 counts both


**What we see.** The memo reported 10,527 people with a dated appearance; the
spine finds 10,425 with a *living* career. The 102 missing are exactly the people who
appear **only** through their estates — never in their own lifetime. The gap is not a
discrepancy but the exclusion doing its job, and those 102 are a distinct historical
category worth keeping in view.

In [7]:
buckets = pd.cut(dated.span_years, [-1, 0, 10, 20, 30, 40, 50, 60, 200],
                 labels=["a single year", "1-10", "11-20", "21-30",
                         "31-40", "41-50", "51-60", "more than 60"])
buckets.value_counts().sort_index().rename("persons").to_frame()

,persons
span_years,
a single year,7729
1-10,1568
11-20,569
21-30,321
31-40,143
41-50,68
51-60,22
more than 60,5


**What we see.** Three quarters of everyone appears within a single year — the
corpus is overwhelmingly made of brief appearances, which is precisely why linkage is
hard here. At the other end, only **5 people** have a career longer than 60 years.

This is worth pausing on, because it revises an earlier estimate. The research memo
counted 28 people with suspiciously long spans; once posthumous rows and undated
contracts are excluded, almost all of those turn out to have been artefacts of the
data rather than genuine over-merges. The real suspect list is tiny — and it means a
60-year career cap is a *sharp* rule, not a blunt one: it will flag almost nothing
that is innocent.

## 5. What counts as network evidence?

Two records that invested alongside the same people are likely the same person. But
that only holds if the shared partner is somebody in particular. A handful of
financiers appear across the whole corpus, and sharing one of them says almost
nothing.

In [8]:
degree = (spine[["person_id", "first_name", "last_name", "contracts"]]
          .assign(n_contracts=lambda d: d.contracts.apply(len))
          .sort_values("n_contracts", ascending=False))
print(f"persons appearing on more than {HUB_CONTRACT_THRESHOLD} contracts: "
      f"{int((degree.n_contracts > HUB_CONTRACT_THRESHOLD).sum())}")
degree.head(6)[["person_id", "first_name", "last_name", "n_contracts"]].set_index("person_id")

persons appearing on more than 20 contracts: 22


,first_name,last_name,n_contracts
person_id,,,
11311,Benedetto,Tempi,60
11308,Francesco,Tempi,60
5155,Gabbriello,Riccardi,42
11477,Francesco,Riccardi,41
12309,Folco,Rinuccini,40
3758,Jacopo,Guadagni,31


**What we see.** Just **22 people** — the Tempi brothers on 60 contracts each, the
Riccardi, Folco Rinuccini — sit far above everyone else. These are excluded from every
partner set, so that "they shared a partner" means they shared an *informative* one.
It is a crude term-frequency correction, and it touches a small, named, reviewable
group rather than silently reshaping the data.

A second correction lives alongside it, added on 2026-08-29. `firm_tokens` holds the
distinctive words of the firms a person traded under — but a Florentine firm is named
after its partners, so tokenising *Gianfigliazzi e Tornabuoni* handed **Leonardo
Gianfigliazzi** his own name back as "network" evidence. Two same-named rows then
intersected on those tokens automatically, and the model counted it as independent
support for their being one man when it was only the name, counted twice. A person's
own chain is now stripped from their own firm tokens; a *partner's* name in the firm
title is exactly the evidence this feature is for, and stays.

Note what is *not* done here: the partner sets are stored as plain lists, and pairs
who appear together on the same contract are dealt with elsewhere. Research showed
that 65% of same-name pairs sharing a contract are one man entered twice, so those
pairs are routed to a human-review lane before any scoring — which is also why their
partner overlap never gets a chance to mislead the model.

## 6. Checkpoints

Every figure this notebook depends on, asserted against the memo. If the corpus
changes, this cell fails and the conclusions above are known to need revisiting.

In [9]:
# All figures: docs/person_linkage/research.md §2, verified against main.db 2026-08-28
assert len(spine) == 11_391,                      "live person count changed"
assert int(spine.is_woman.sum()) == 242,          "women count changed"
assert int((spine.n_appearances == 0).sum()) == 856, "zero-appearance ghosts changed"
assert len(dated) == 10_425 and len(heirs_only) == 102
assert 56 <= 100 * (spine.patronymic_norm != "").mean() <= 58, "patronymic fill drifted"
assert 12 <= 100 * (spine.grandfather_norm != "").mean() <= 14, "grandfather fill drifted"

# No missing value may masquerade as a name (see tests/test_person_features.py)
for column in ["first_norm", "patronymic_norm", "grandfather_norm", "last_norm"]:
    assert not (spine[column] == "nan").any(), f"{column} contains the literal string 'nan'"

# The cast behaves as the research described
assert spine.set_index("person_id").loc[1164, "span_years"] == 69   # the over-merge
assert spine.set_index("person_id").loc[3460, "span_years"] == 0    # posthumous excluded
print("all checkpoints passed")

all checkpoints passed


## 7. Not everyone in the table is a person

Before comparing people, we should check that we are comparing people. The `person`
table also holds institutions that invested, estates acting for the dead, and a few
placeholders the clerks used when a name was not available.

In [10]:
kinds = spine.entity_kind.value_counts().rename("rows").to_frame()
kinds["appearances"] = spine.groupby("entity_kind").n_appearances.sum()
kinds

,rows,appearances
entity_kind,,
person,11330,17416
placeholder,37,33
institution,15,26
estate,7,1
collective,2,19


In [11]:
# the rows that stand for MANY people rather than one
collectives = spine[spine.entity_kind == "collective"]
collectives[["person_id", "first_name", "last_name", "n_appearances",
             "first_year", "last_year"]].set_index("person_id")

,first_name,last_name,n_appearances,first_year,last_year
person_id,,,,,
2797,azionisti vari,NaN,1,1767.0,1767.0
12254,,MOLTEPLICI AZIONARI,18,1788.0,1807.0


**What we see.** Sixty-one rows are not individual people, and one of them is
remarkable. A row recorded as **MOLTEPLICI AZIONARI** — *multiple shareholders* —
appears on **eighteen contracts**, which would make it one of the busiest "investors"
in the corpus.

It is not sloppiness. All eighteen of its narratives speak of *azioni*, *azionisti*,
*azionari* or the Livornese *carati*, the firms include the *Nuova Compagnia
d'Assicurazioni* and marine insurers, and the contracts fall in **1788–1807** — the
corpus's last two decades. By then the accomandita is being used as a **share
company** whose partners are too many and too anonymous to name, and this placeholder
is the only trace of that in the structured data. Every one of its appearances is
recorded as a *limited* partner, exactly as a shareholder should be.

So this is a historical finding as much as a data problem — written up in
[docs/data_quality/non_person_rows.md](../docs/data_quality/non_person_rows.md), with
two questions for the PI in the decisions register. For our purposes here, these rows
simply cannot be linked as people, so they are set aside before matching begins.

## 8. What a rule can decide — and what it must not

Some pairs need no model. If two rows have the same name and one of them never appears
in any contract, that is a data-entry ghost. If merging two rows would imply a career
longer than any working life, they are two people. The tiers apply those rules and
leave everything else alone.

In [12]:
from workflows.person_tiers import build_tiers, combined_span, classify_pair

tiers = build_tiers(spine)
counts = tiers.tier.value_counts().rename("pairs").to_frame()
counts["what a human does"] = {
    "distinct_strong": "accept the refusal in bulk",
    "review": "→ the model ranks these",
    "caution_gf_conflict": "read the act",
    "caution_coappearance": "read the act",
    "batch_ghost": "confirm the duplicate",
    "same_as_strong": "confirm the match",
}
counts

,pairs,what a human does
tier,,
review,1886,→ the model ranks these
distinct_strong,873,accept the refusal in bulk
caution_gf_conflict,67,read the act
caution_coappearance,31,read the act
batch_ghost,15,confirm the duplicate
same_as_strong,2,confirm the match


**What we see.** Of 2,874 candidate pairs, rules settle **890** outright and refuse
to guess at 98 more; the remaining **1,886** go to the model. The human workload is
about **115 pairs needing real attention** (the 98 cautions, plus 15 batch ghosts
and 2 confirmed matches to sign off) plus a bulk acceptance — a workable first
sitting.

Note which lanes deliberately decide *nothing*. Two same-named people on one
contract look like proof they are different men; in this corpus they are the same
man entered twice 65% of the time. That is a caution with the document attached,
never a verdict — and §9 shows what the model does with those pairs, which is
deliberately nothing at all.

### The rule that had to be rewritten

The first version of the career rule said "same name, careers less than 40 years apart
⇒ same man". The Torrigiani family disproves it without any outside evidence.

In [13]:
torrigiani = spine[spine.person_id.isin([761, 12307, 12308])]
display(torrigiani[["person_id", "first_name", "father_mother", "grandfather",
                    "last_name", "first_year", "last_year"]].set_index("person_id"))

rows = spine.set_index("person_id")
for l, r in [(761, 12307), (12307, 12308), (761, 12308)]:
    a, b = rows.loc[l], rows.loc[r]
    gap = int(max(b.first_year - a.last_year, a.first_year - b.last_year))
    span = int(max(a.last_year, b.last_year) - min(a.first_year, b.first_year))
    print(f"  {l} vs {r}:  gap between careers {gap:>3} years   ·   "
          f"one life would need {span:>3} years")

,first_name,father_mother,grandfather,last_name,first_year,last_year
person_id,,,,,,
761,Luca,Raffaello,Luca,Torrigiani,1546.0,1549.0
12307,Luca,Raffaello,Luca,Torrigiani,1572.0,1595.0
12308,Luca,Raffaello,Luca,Torrigiani,1638.0,1638.0


  761 vs 12307:  gap between careers  23 years   ·   one life would need  49 years
  12307 vs 12308:  gap between careers  43 years   ·   one life would need  66 years
  761 vs 12308:  gap between careers  89 years   ·   one life would need  92 years


**What we see.** Judged on the **gap**, the first two pairs (23 and 43 years) both
look mergeable while the third (89 years) cannot be — so the rule would say A=B and
B=C but A≠C. That is not a rule.

Judged on the **career one person would need** — 49, 66 and 92 years — the verdicts
agree, and they agree for a reason: merging more rows can only lengthen a life, so the
measure can never contradict itself. This is what the tiers use, and it is why
761≈12307 is proposed as a match while the other two are refused.

## 9. Teaching a model to rank the rest

The 1,886 undecided pairs need judgement, not rules. We train a Splink model —
probabilistic record linkage — to weigh **five** kinds of evidence: the name, the
patronymic chain, **contemporaneity** (the implied career and the business network,
as one comparison — see below for why they cannot be two), the partnership role,
and, for women, the husband's name.

Training takes a few minutes: it learns how often fields agree *by coincidence* by
comparing every possible pair, then how often they agree *for true matches* from
within the name blocks.

**A note on which pairs get looked at at all.** We cannot compare all 64 million
possible pairs, so the model makes several passes over the records — sorting by
first name and surname, by surname and father, and so on — and anyone who lands
together in a pass gets properly compared. Three of those four passes sort on the
**surname**, so a surname written two ways falls through all of them: *Salvatore
Innori di Vincenzo* (1588) and *Salvadore Inori di Vincenzo* (1589) share a father
**and** a grandfather and were never compared at all. A fifth pass now sorts by the
father's name and, within each pile, compares anyone whose surnames are within one
letter. Requiring the father to agree is what keeps it honest — surname resemblance
alone puts *Gucci* with *Pucci* and *Corbini* with *Corsini*, distinct houses that
merely look alike. It costs 406 extra comparisons out of ~15,600 and finds the
Innori pair at **p=0.89**. It also reaches a *Brandolini*/*Barandolini* pair, which
scores only **0.13** — worth stating plainly, because the lane widens the field of
view without lowering the bar.

**Read this section knowing that the first model built here was wrong.** It was
audited on 2026-08-29 and three of its six comparisons turned out to hold parameters
that were not probabilities at all — they did not sum to 1 — which had inflated the
results by roughly a third. Nothing in this notebook caught it, because every
checkpoint above tests the *data* and none tested the *model*. The repair, the
mechanism and the guards now in place are described as they come up below, and the
model is rebuilt by `scripts/train_person_model.py`, which refuses to save one that
fails the check.

In [14]:
from workflows.person_model import prepare_frame, train_model, check_model_is_well_formed

# Splink's warnings are NOT silenced here. An earlier version of this cell set the
# splink logger to ERROR, which hid "u values not fully trained" on every predict()
# — the one signal that the shipped model was malformed. See the checkpoint below.
model_input = prepare_frame(spine)
linker = train_model(model_input)
print(f"trained on {len(model_input):,} people")

# CHECKPOINT — the model, not just the data. The 2026-08-28 model failed this and
# nothing noticed for a full cycle, because every figure anyone quoted was derived
# from that same file. m and u are probabilities over the levels of one comparison,
# so each must sum to 1.
faults = check_model_is_well_formed(linker.misc.save_model_to_json())
assert not faults, "model is not a valid Fellegi-Sunter model: " + "; ".join(faults)
print("model well-formed: every m and u vector sums to 1")

trained on 11,330 people
model well-formed: every m and u vector sums to 1


In [15]:
linker.visualisations.match_weights_chart()

/Users/wouter/gitrepos/floracco/.venv/lib/python3.12/site-packages/altair/vegalite/v6/api.py:4138: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

**What we see.** Each bar is evidence: to the right argues that two rows are the
same person, to the left that they are not. Nobody told the model what should
matter, and yet it puts the patronymic chain at **+10.76** when father and
grandfather both agree, and **being in the same firm at +10.05** — the strongest
network signal of all, above sharing three or more business partners at +8.14.

One bar looks bigger than either and should be read carefully: the name's own
exact-match level scores **+11.52**. But almost every pair we score already shares a
name, because that is how the candidates were selected — so that bar is a property
of the blocking, not a finding. The patronymic chain and the firm are the strongest
evidence the selection did *not* hand us.

**Five comparisons, not six.** `career` and `network` used to be separate. They
cannot be. Fellegi–Sunter multiplies each comparison's weight *as if they were
independent given match status*, and these two are anything but: you cannot share a
business partner with someone who died before you were born. Measured on 2.7
million random pairs, P(share a partner | careers overlap) = **4.8%** against
**0.12%** when they do not — a 40× ratio, **+4.35 bits** of double-counted evidence,
paid on the model's commonest high-scoring pattern. They are now one comparison
whose levels are joint events, so the algorithm prices the conjunction once.

**The firm is the newest evidence and the most interesting.** An accomandita is a
fixed-term partnership that gets *renewed* — 353 firm names in this corpus span 821
contracts, *Mario Morelli e compagni* across eight — so the same people reappear
together act after act. That matters because it is the one way around a wall the
partner evidence cannot climb: `partners` is a list of *person ids*, and in a corpus
whose whole problem is duplicated people, a duplicated person's partners are
duplicated too. Luigi Capponi's two records share no partner id at all, because his
partner Alessandro is himself split in two. Resolving that would need identity —
which is the question being asked. **A firm name is just a string.** It needs no
resolving.

Two levels carry it, and the second was not obvious. "Contemporaries, the same
firm" (+10.1) was the first attempt, and it did nothing for the cases it was built
for, because their careers *abut* rather than overlap — one record ends in 1591, the
next begins in 1592 — and they already had generic network credit from two shared
firm-name words. "Career ≤30 years, the same firm" (**+9.8**) fixed that. Records
that abut in time and share a firm are a career continuing, not two men.

Things in this chart that were *imposed* rather than learned, each for a reason:

1. **The sibling signature** (−10.1) and **father-and-son signature** (−1.1) are
   fixed by hand. Left free, the algorithm learned father-and-son as **+8 bits in
   favour** of a match, because it cannot see that a man is not his own father.
2. **The husband disagreement** (−3.3) is also asserted. Left to EM it collapsed to
   **−45.9 bits** — a veto strong enough to annihilate any pair however much else
   agreed — learned from a comparison that is non-null on just 14 of 15,637 pairs.
   It sat on the one comparison serving the 242 women in the corpus.
3. **"Contemporaries" carries its own span bound.** Splink takes the first matching
   level, so an unqualified overlap test outranked every "career ≤ N years" level
   beneath it — reading the Florentine succession, a son entering the firm while his
   father still trades, as proof the two are one man.
4. **A shared act makes the whole comparison say nothing.** Two rows in the same
   document share every other signatory mechanically and their dates coincide by
   construction, so both halves would restate that one document.
5. **A father's name one letter apart** (+3.9) is priced by the algorithm rather
   than by a hand-written table of spellings — see §9's note.

## 10. Does the model agree with the rules?

The model never saw the tiers. So comparing them is a real test: if the two disagree
wildly, one of them is wrong.

In [16]:
pred_sdf = linker.inference.predict()
pred = pred_sdf.as_pandas_dataframe()
score = {tuple(sorted(p)): w for p, w in
         zip(zip(pred.person_id_l, pred.person_id_r), pred.match_weight)}
tiers["match_weight"] = [score.get((min(l, r), max(l, r)))
                         for l, r in zip(tiers.person_id_l, tiers.person_id_r)]
tiers.groupby("tier").match_weight.agg(["count", "median", "min", "max"]).round(1)

,count,median,min,max
tier,,,,
batch_ghost,15,1.2,-2.1,2.2
caution_coappearance,31,-9.4,-10.8,-1.3
caution_gf_conflict,67,-7.0,-12.1,5.4
distinct_strong,873,-10.5,-13.9,1.9
review,1886,-8.1,-12.8,9.7
same_as_strong,2,5.3,1.3,9.3


**What we see.** They agree, and they agree in the right direction: pairs the rules
refused sit at a median of about **−10**, pairs the rules confirmed at **+5.8**.
Exactly **one** of the 873 refusals outscores the weakest confirmed match — and that
weakest match is itself a marginal case (Torrigiani 761/12307 at **+1.34**), so the
overlap is narrower than a single number suggests.

**The co-appearance cautions now sit at a median of −9.4, and that is the design
working rather than a disagreement.** An earlier version of this notebook reported
them as *positive*, and reasoned that the model independently leaned "this is one
man". That reading was wrong, and the audit of 2026-08-29 explains why: for 15 of
the 17 such pairs then scoring above 0.90, **every** shared partner came from the
single contract the pair both appear on. `partners` is built from contract
co-parties, so two rows on one act share every other signatory *mechanically*, and
their dates coincide *by construction*. The model was not corroborating the rule; it
was restating the one fact that triggered it, three times over.

A shared act therefore now makes the whole time-and-network comparison say nothing.
These pairs fall back on name and lineage alone, which is honest — and they go to a
human either way, with the document attached. That is what the caution lane is for.

In [17]:
top = tiers[(tiers.tier == "review") & tiers.match_weight.notna()].nlargest(8, "match_weight")
top[["person_id_l", "person_id_r", "name", "match_weight"]].set_index("person_id_l")

,person_id_r,name,match_weight
person_id_l,,,
10873,11331,Maria Cerretani,9.711864
974,12143,Lodovico da Verrazzano,9.340966
1586,11824,Riccardo Riccardi,9.340966
1951,12153,Giovanni Cerretani,9.340966
1088,11907,Agostino del Nero,9.064803
4756,10976,Luca Franceschi,8.756004
4757,10975,Agostino Franceschi,8.756004
1810,11369,Giovanni Franchi,8.479840


**What we see.** The top of the review lane is led by the **Lanfranchi** pairs — and
those are the same pairs the research phase picked out by hand weeks ago as clear
merges (same firm, three shared partners). The model found them on its own, which is
about as good a sanity check as this project can get.

### Reading a single decision

The point of this model is not the score but the explanation. A waterfall chart shows
exactly which evidence moved each pair, and by how much.

In [18]:
cases = [(548, 11821), (1591, 12025), (818, 12322)]
wanted = pred[[ (min(l, r), max(l, r)) in cases
                for l, r in zip(pred.person_id_l, pred.person_id_r) ]]
linker.visualisations.waterfall_chart(wanted.to_dict(orient="records"), filter_nulls=False)

alt.LayerChart(...)

**What we see.** Three very different stories:

- **Minerbetti 548/11821** — every piece of evidence points the same way: the full
  name chain, an overlapping career, a shared network. An easy confirmation.
- **Lanfranchi 1591/12025** — the name is common enough to be weak on its own, and
  the *network* carries the decision. This is the pair the research phase picked out
  by hand weeks earlier; the model found it unaided.
- **Corsi 818/12322** — and here the story has changed. This pair used to score
  **+6.3** while the rules refused it, and it was the worked example of why a score
  must never overturn a deterministic verdict. After the repairs of 2026-08-29 it
  scores **−1.02**: the model now refuses what the rule refuses.

That does not retire the precedence rule, it removes one instance of the problem it
exists for. A score may rank what is undecided; it may never resurrect a pair that
time has already excluded, whether or not any pair currently tries to. And the
group-level guard still matters, because no pairwise precedence can reach a chain of
individually defensible links that assembles into someone who never lived.

## 11. From pairs to people

A pairwise score is not yet an answer. The question the project asks is *how many
people are there*, which means turning links into groups — and groups are where a
model's mistakes become visible, because a chain of individually plausible links can
describe a person who never lived.

First, a warning about reading the scores. Almost every pair we score already shares
a name, because that is how the candidates were selected. So the name comparison
contributes about +11.6 to most pairs, against a prior of −19.2: the neutral point
sits near **−7.5**, not at zero. A weight of 0 is already substantial evidence, not a
coin flip. (The exception is the surname-variant lane from §9, where the surnames
deliberately differ and the pairs start lower.)

In [19]:
for t in (0, 2, 4, 6, 8, 10):
    n = int((pred.match_weight >= t).sum())
    print(f"  weight >= {t:>2}  →  {n:>5,} pairs   (probability >= {1/(1+2**-t):.3f})")

  weight >=  0  →    216 pairs   (probability >= 0.500)
  weight >=  2  →    100 pairs   (probability >= 0.800)
  weight >=  4  →     58 pairs   (probability >= 0.941)
  weight >=  6  →     26 pairs   (probability >= 0.985)
  weight >=  8  →     12 pairs   (probability >= 0.996)
  weight >= 10  →      0 pairs   (probability >= 0.999)


**What we see.** The scores fall away steeply: 216 pairs clear the halfway mark,
58 reach 94%, 12 reach 99.6%, and none reach 99.9%.

These numbers are far smaller than this notebook printed earlier in the day. Two
changes account for it — removing the double-counted time-and-network evidence, and
making a shared act carry no weight — and together they took proposed groups from
140 to **64**. Two independent signs say the smaller number is the better one.
First, the model's own prior implies about **110** true matched pairs in the whole
corpus, so 163 was never credible and 64 is. Second, the deterministic rules and the
model now agree completely: **not one** of the 873 pairs the rules refuse outright
still scores above 0.90, where three did before.

The model still overshoots its own prior — posteriors sum to about 324 against 110 —
so these remain rankings rather than calibrated probabilities. That figure prints on
every training run, and it is a **lower bound**, since it compares a corpus-wide
prior against a posterior sum over blocked pairs only. Read the other way, it says
our assumed `recall = 0.5` for the deterministic rules was probably too generous:
the gap closes at about 0.18, which is plausible for rules demanding a complete
four-part name chain in a corpus where 87% lack a grandfather.

In [20]:
from workflows.person_model import flag_impossible_clusters

clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    pred_sdf, threshold_match_probability=0.90).as_pandas_dataframe()

sizes = clusters.groupby("cluster_id").size()
multi = sizes[sizes > 1]
audit = flag_impossible_clusters(clusters, spine)

# Report the two counts separately. `flag_impossible_clusters` silently skips any
# cluster with fewer than two DATED members — it cannot judge what it cannot date —
# so len(audit) is the judgeable subset, not the group count. They happen to be
# equal on today's data; printing len(audit) as if it were the group count would
# start lying the moment an undated cluster appeared.
print(f"groups of 2+ rows at p>=0.90 : {len(multi)}  (covering {int(multi.sum())} person rows)")
print(f"  datable, so judgeable      : {len(audit)}")
print(f"  implying a career > 60 yrs : {int(audit.impossible.sum())}")
audit.head(5)[["size", "first_year", "last_year", "implied_career_years", "impossible"]]

groups of 2+ rows at p>=0.90 : 64  (covering 128 person rows)
  datable, so judgeable      : 64
  implying a career > 60 yrs : 0


,size,first_year,last_year,implied_career_years,impossible
0,2,1683,1743,60,False
1,2,1559,1613,54,False
2,2,1623,1673,50,False
3,2,1568,1613,45,False
4,2,1574,1616,42,False


**What we see, and why it matters.** Clustering exposes a failure that pairwise
scores cannot: connected components merge A–B and B–C into one identity, so plausible
links can chain into an impossible person.

The clearest case was the Corsi. Unguarded, the model gathered **eleven** rows into a
single 101-year "person" — and reading the names shows why: *Bardo*, *Giovanni*,
*Simone* and *Lorenzo* **di Jacopo Corsi** are **brothers**, named as such in the acts
themselves. Siblings share a father, a surname and an entire business network;
everything the model looks at agrees except the given name.

**No group now implies an impossible career at all** — down from eleven rows in one
person, to six such groups this morning, to two, to none. The last two did not
dissolve because of a threshold: the Corsi group was being held together by two
edges whose *entire* shared-partner evidence came from a single act both rows
appeared in. Once a shared act makes the time-and-network comparison say nothing,
those edges fell from p≈0.90 to 0.09 and 0.07 and the group came apart.

That is worth pausing on, because it is the opposite of the usual story about
cleaning up a model. The over-merges were not noise to be thresholded away. They
were **the predictable consequence of counting one piece of evidence twice**, and
they disappeared when the double-counting did.

`flag_impossible_clusters` stays regardless — it is the pair-level career rule
lifted to whole groups, and a chain of individually defensible links can still
describe someone who never lived. It needs two *dated* members to judge a group, so
the cell above prints what it could judge separately from the group count.

### What we deliberately did *not* add

Three things were considered and rejected, and the reasons are worth recording:

- **Titles** (*messer*, *illustrissimo*) — 7% of people who are certainly one person
  show a *drop* in title over their career. Scribal variation, not evidence.
- **Residence and profession** — the fields look full but 83% of the values are a
  sentinel meaning "not recorded". Scoring agreement there would be scoring the
  sentinel.
- **Religious attribution** — excluded on both ethical and statistical grounds:
  community-distinctive names already carry that signal through the
  term-frequency adjustment, so including it would double-count while attaching a
  sensitive inference to an automated score.

## Where this leaves us

Stage 1 is complete: from 11,391 rows we have 11,330 people, a tiered worklist of 2,874
candidate pairs, and a trained model that ranks the undecided ones with an explanation
attached to every score.

What was learned along the way is as valuable as the output:

1. **Missing data nearly became a name** — empty patronymics arrived as the literal text
   `nan`, which would have made every pair *lacking* a patronymic appear to agree.
2. **The over-merge suspect list shrank from 28 to 5** once posthumous and undated rows
   were excluded.
3. **A placeholder row turned out to be a historical finding** — the share companies of
   1788–1807.
4. **The first career rule was not transitive**, and the Torrigiani family proved it.
5. **The model learned one level backwards** and had another silently untrained; both
   are now pinned by tests.
6. **The model itself was never checked, and it was broken** — see below.

**The audit of 2026-08-29.** The model this notebook shipped on 28 August was not a
valid Fellegi–Sunter model. Three of its six comparisons held `m` values that did not
sum to 1, which is to say they were not probabilities; the name comparison asserted
that a true match was as likely to carry a completely different name as an identical
one. Repairing it removed about a third of the proposed groups. Four things let it
survive, and each has a guard now:

- **Every checkpoint in this notebook tested the data, none tested the model.** The
  spine was asserted line by line; the artifact derived from it was asserted not at
  all. Section 9 now ends with a well-formedness assertion.
- **The one warning that would have caught it was being suppressed.** Splink printed
  "u values not fully trained" on every prediction, and both this notebook and the
  dashboard script had set the splink logger to ERROR.
- **There was no script to rebuild the model** — it existed only as a hand-run cell, so
  there was no reproducible path and nothing standing between a bad fit and the repo.
  `scripts/train_person_model.py` now refuses to save a model that fails the check.
- **A stale line in the memo said the name comparison "carries no weight"**, which made
  it the last place anyone would look. It was true of one blocking rule out of four.

The general lesson is the uncomfortable one: *auditable* is not *audited*. The model
was a versioned JSON artifact in the repository for a full development cycle, and every
number anyone quoted from it — here, in the memo, in the log, in the dashboard — was a
property of a file no one had checked.

**What this cannot do yet.** The model's parameters come from the algorithm's own
guesses, not from human judgement, because the deterministic lanes yield only ~10
confident positive examples — far too few to train on. That is structural: asserting
*sameness* by rule is exactly what the Florentine naming custom prevents. And the model
overshoots its own prior by about five times, so its probabilities rank well but should
not be read as calibrated.

**So the next step is people, not code.** When historians work the ~108 rule-decided
pairs and the top of this ranking, their confirmations become the first real training
labels, and the model can be re-estimated on human judgement instead of on inference.
That is the right order for this project: the scholarship trains the machine.